# CrediFast Phase 3: history feature quality and model comparison

## tl;dr

- The target-free history store reconciles all **356,255** train and test applications at one row per `SK_ID_CURR`; the automated quality gate passes with no invalid rates, negative counts, or non-finite values.
- History coverage ranges from **29.1% for credit cards** to **95.3% for installments**. Missing source blocks are therefore modeled with explicit availability flags, not converted to zero behavior.
- On the untouched calibration partition, the history-enriched LightGBM reaches **0.7814 ROC-AUC** and **0.2739 average precision**, improving over the application-only LightGBM while the final holdout remains sealed.


## Context & Methods

This notebook is the inspectable audit trail for Phase 3. It reads the saved aggregation, quality, and model metric artifacts produced by repository commands; it does not retrain models or alter data.

### Key Assumptions

- All six history tables describe events observed before the current application decision.
- A missing source history is not equivalent to a measured zero balance or zero delinquency.
- Model comparisons use the same seed and 70/15/15 stratified split. Only the calibration partition is scored here.


## Data

### 1. Load saved evidence

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"

quality = json.loads((ARTIFACTS / "history_quality_report.json").read_text())
feature_report = json.loads((ARTIFACTS / "history_features_report.json").read_text())
quality["summary"]


{'row_count': 356255,
 'column_count': 151,
 'feature_count_excluding_key': 150,
 'unique_keys': 356255,
 'null_keys': 0,
 'expected_application_rows': 356255,
 'maximum_finding_severity': 'none',
 'ready_for_modeling': True}

### 2. Review source coverage

In [2]:
coverage = (
    pd.DataFrame(quality["checks"]["source_coverage"])
    .T.rename_axis("source")
    .reset_index()
)
coverage["coverage_percent"] = 100 * coverage["rate"]
coverage[["source", "rows", "coverage_percent"]].sort_values("coverage_percent")


,source,rows,coverage_percent
3,HAS_CREDIT_CARD_HISTORY,103558.0,29.068504
0,HAS_BUREAU_HISTORY,305811.0,85.840479
2,HAS_POS_HISTORY,337252.0,94.665899
1,HAS_PREVIOUS_APPLICATION,338857.0,95.116419
4,HAS_INSTALLMENT_HISTORY,339587.0,95.321329


## Results

### 3. Confirm the automated modeling-readiness gate

These are hard invariants: exact application reconciliation, unique/non-null primary keys, no target field, bounded rate features, nonnegative counts, and finite numeric values.


In [3]:
assert quality["automated_gate"]["passed"]
assert quality["summary"]["ready_for_modeling"]
assert quality["summary"]["row_count"] == quality["summary"]["expected_application_rows"]
assert quality["summary"]["unique_keys"] == quality["summary"]["row_count"]
pd.DataFrame({"required_check": quality["automated_gate"]["required"]})


,required_check
0,row count equals application_train plus applic...
1,SK_ID_CURR is non-null and unique
2,TARGET is absent
3,all count features are nonnegative
4,all rate features are within zero and one
5,all populated numeric values are finite


### 4. Quantify source-availability differences

Availability itself separates populations, so the `HAS_*` indicators are operationally meaningful. These rates are descriptive associations only and are not causal fairness claims.


In [4]:
availability_rows = []
for source, groups in quality["checks"]["target_rate_by_source_availability"].items():
    for group in groups:
        availability_rows.append({"source": source, **group})
availability = pd.DataFrame(availability_rows)
availability["target_rate_percent"] = 100 * availability["target_rate"]
availability[["source", "available", "rows", "target_rate_percent"]]


,source,available,rows,target_rate_percent
0,HAS_BUREAU_HISTORY,0,44020,10.124943
1,HAS_BUREAU_HISTORY,1,263491,7.730055
2,HAS_PREVIOUS_APPLICATION,0,16454,5.955999
3,HAS_PREVIOUS_APPLICATION,1,291057,8.192553
4,HAS_POS_HISTORY,0,18067,6.664084
5,HAS_POS_HISTORY,1,289444,8.160819
6,HAS_CREDIT_CARD_HISTORY,0,220606,7.837955
7,HAS_CREDIT_CARD_HISTORY,1,86905,8.669237
8,HAS_INSTALLMENT_HISTORY,0,15868,5.980590
9,HAS_INSTALLMENT_HISTORY,1,291643,8.186721


### 5. Compare models on the same calibration partition

In [5]:
metric_files = [
    "application_baseline_metrics.json",
    "application_lightgbm_metrics.json",
    "history_lightgbm_metrics.json",
]
comparison_rows = []
for metric_file in metric_files:
    artifact = json.loads((ARTIFACTS / metric_file).read_text())
    metrics = artifact["calibration_partition_metrics"]
    comparison_rows.append(
        {
            "model": artifact["model_name"],
            "roc_auc": metrics["roc_auc"],
            "average_precision": metrics["average_precision"],
            "brier_score": metrics["brier_score"],
            "top_10_capture": metrics["top_10_percent"]["event_capture_rate"],
            "top_20_capture": metrics["top_20_percent"]["event_capture_rate"],
            "holdout_evaluated": artifact["holdout_evaluated"],
        }
    )
comparison = pd.DataFrame(comparison_rows)
comparison.round(4)


,model,roc_auc,average_precision,brier_score,top_10_capture,top_20_capture,holdout_evaluated
0,application-logistic-baseline,0.7459,0.2326,0.0685,0.3241,0.5019,False
1,application-lightgbm-challenger,0.7588,0.2492,0.0676,0.3416,0.5193,False
2,history-enriched-lightgbm-challenger,0.7814,0.2739,0.0663,0.3673,0.5588,False


In [6]:
application_only = comparison.loc[
    comparison["model"].eq("application-lightgbm-challenger")
].iloc[0]
history_enriched = comparison.loc[
    comparison["model"].eq("history-enriched-lightgbm-challenger")
].iloc[0]
pd.Series(
    {
        "roc_auc_delta": history_enriched.roc_auc - application_only.roc_auc,
        "average_precision_delta": (
            history_enriched.average_precision - application_only.average_precision
        ),
        "brier_score_delta": history_enriched.brier_score - application_only.brier_score,
        "top_20_capture_delta": (
            history_enriched.top_20_capture - application_only.top_20_capture
        ),
    },
    name="history_minus_application_only",
).round(4)


roc_auc_delta              0.0226
average_precision_delta    0.0247
brier_score_delta         -0.0013
top_20_capture_delta       0.0395
Name: history_minus_application_only, dtype: float64

## Takeaways

1. The history feature store is safe to use for model development under the automated gate and preserves the intended application grain.
2. Sparse credit-card history is structural, not a pipeline failure; source blocks must retain nulls plus availability flags.
3. History improves both ranking and rare-event precision on calibration data. The model is a challenger, not yet a production credit decision system.
4. Next: calibrate probabilities, freeze operating bands and reason-code behavior, then score the sealed holdout once.
